## Tools

Models can request to call toold that perform tasks such as fetching from a databse , searching the web , or runnin gocde . Tools are pairings of :
   
   1. A schema , including the name of the tool , a description , and /or arguement definitions (often JSON      schema)
   2.A function or coroutine to execute .

In [1]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")



In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"it's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [3]:
response = model_with_tools.invoke("whats the weather like in Delhi?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tools: {tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Delhi. I need to use the get_weather function. The function requires a location parameter. Delhi is the location here. So I should call get_weather with location set to "Delhi". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'v5rhz3mty', 'function': {'arguments': '{"location":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 154, 'total_tokens': 250, 'completion_time': 0.162827338, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.007368886, 'prompt_tokens_details': None, 'queue_time': 0.157619728, 'total_time': 0.170196224}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_ru

## Tool Execution Loops

In [4]:
messages = [{"role":"user","content":"Whats the weather in Delhi?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)    

The weather in Delhi is sunny today.


In [5]:
messages

[{'role': 'user', 'content': 'Whats the weather in Delhi?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Delhi. I need to use the get_weather function. The function requires the location parameter. Delhi is the location here. So I should call get_weather with location set to "Delhi". Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'zq8f1ck14', 'function': {'arguments': '{"location":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 152, 'total_tokens': 247, 'completion_time': 0.15786836, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.006555882, 'prompt_tokens_details': None, 'queue_time': 0.05908501, 'total_time': 0.164424242}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason